<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/TIC-TAC-TOE_lektioner/Lab_2_Lektion_3_Slumpmassig_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Tre i rad – Lektion 3: Slumpmässig AI-agent

**Målgrupp:** Gymnasiet, inga förkunskaper i programmering krävs  
**Tid:** ca 45 minuter  
**Mål:** Förstå vad en AI-agent är, bygga en slumpmässig agent, spela mot datorn och analysera resultaten

---

**Upphovspersoner:** LiU / Tekniksprånget  
**Licens:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.sv)

---

### �� Förutsättning
Den här lektionen bygger vidare på **Lektion 1 och 2**.  
Se till att du förstår hur `check_winner`, `make_move` och det interaktiva spelet fungerar.

### 🗺️ Vad gör vi idag?
1. Lär oss vad en **agent** är
2. Bygger en funktion som hittar **tomma celler**
3. Skapar vår **första datoragent** – en slumpmässig agent
4. Spelar mot datorn! 🤖
5. Låter datorn spela mot sig självt och räknar statistik


In [ ]:
# ============================================================
# INSTÄLLNINGAR – Kör denna cell FÖRST innan du fortsätter!
# Den innehåller all spellogik från tidigare lektioner.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import random   # Behövs för den slumpmässiga agenten

# ----------------------------------------------------------
# Spelplanens storlek – alltid 3x3 i Tre i rad
# ----------------------------------------------------------
BOARD_SIZE = 3

# ----------------------------------------------------------
# check_winner – kollar om någon spelare har vunnit
# Returnerar: 1 om X vann, 2 om O vann, 0 om oavgjort/pågående
# ----------------------------------------------------------
def check_winner(board):
    """Kontrollerar om det finns en vinnare på spelplanen."""
    # Kolla alla rader (horisontellt)
    for rad in range(BOARD_SIZE):
        if board[rad][0] != 0 and board[rad][0] == board[rad][1] == board[rad][2]:
            return board[rad][0]  # Returnera vinnaren (1 eller 2)

    # Kolla alla kolumner (vertikalt)
    for kol in range(BOARD_SIZE):
        if board[0][kol] != 0 and board[0][kol] == board[1][kol] == board[2][kol]:
            return board[0][kol]

    # Kolla diagonal uppifrån-vänster till nerifrån-höger
    if board[0][0] != 0 and board[0][0] == board[1][1] == board[2][2]:
        return board[0][0]

    # Kolla diagonal uppifrån-höger till nerifrån-vänster
    if board[0][2] != 0 and board[0][2] == board[1][1] == board[2][0]:
        return board[0][2]

    # Ingen vinnare ännu
    return 0

# ----------------------------------------------------------
# make_move – försöker placera en spelare på (rad, kol)
# Returnerar True om draget lyckades, False annars
# ----------------------------------------------------------
def make_move(board, row, col, player):
    """Gör ett drag på spelplanen om rutan är tom."""
    if board[row][col] == 0:   # 0 betyder tom ruta
        board[row][col] = player
        return True
    return False  # Rutan var redan tagen

# ----------------------------------------------------------
# visa_braede – skriver ut spelplanen i terminalen (för test)
# ----------------------------------------------------------
def visa_braede(board):
    """Skriver ut spelplanen som text i terminalen."""
    symboler = {0: ".", 1: "X", 2: "O"}
    print("  0 1 2")   # Kolumnnummer
    for rad in range(BOARD_SIZE):
        rad_text = f"{rad} "  # Radnummer
        for kol in range(BOARD_SIZE):
            rad_text += symboler[board[rad][kol]] + " "
        print(rad_text)
    print()

print("✅ Alla funktioner är laddade! Du kan börja lektionen.")


---
## 🤖 Del 1 – Vad är en agent?

### Definition
> **En agent är ett program som observerar sin omgivning och tar beslut.**

I Tre i rad handlar det om att **välja ett drag** – vilken ruta ska man spela på?

### Två typer av agenter vi känner till:

| Typ | Vem bestämmer? | Exempel |
|-----|---------------|---------|
| 🧑 **Mänsklig agent** | En riktig person klickar på en knapp | Du i Lektion 2 |
| 🤖 **Datoragent** | En funktion i koden väljer automatiskt | Det vi bygger nu |

### Vår första datoragent: den slumpmässiga agenten

En **slumpmässig agent** (eng. *random agent*) väljer ett drag helt på måfå.  
Den vet ingenting om strategi – den väljer bara en ledig ruta av misstag, utan tanke.

Låter det dumt? Kanske! Men det är ett utmärkt **startpunkt** för att förstå hur agenter fungerar.  
Senare lektioner bygger smartare agenter. 🧠

### Hur fungerar en slumpmässig agent?
1. Titta på spelplanen
2. Hitta alla **tomma rutor**
3. Välj en av dem **på slump**
4. Gör det draget

Enkelt! Nu kodar vi det.


---
## 🔍 Del 2 – Hitta tomma celler: `find_empty_cells()`

Innan agenten kan välja ett drag måste den veta **var det finns lediga rutor**.

Vi skapar en hjälpfunktion som letar igenom hela spelplanen och samlar ihop alla tomma positioner.

### Hur fungerar det?
Vi loopar igenom **varje rad** och **varje kolumn**.  
Om värdet i cellen är `0` (dvs. tom), lägger vi till positionen `(rad, kol)` i en lista.

```
Spelplan:          Tomma celler:
1 0 2              (0,1) ← rad 0, kolumn 1
0 1 0     →→→     (1,0) ← rad 1, kolumn 0
2 0 0              (1,2), (2,1), (2,2)
```

Positionen sparas som ett **par** (en tuple): `(rad, kol)`.  
Det gör det enkelt att skicka direkt till `make_move`!


In [ ]:
# ============================================================
# find_empty_cells – hittar alla tomma rutor på spelplanen
# ============================================================

def find_empty_cells(board):
    """
    Hittar alla tomma celler på spelplanen.

    Argument:
        board: Spelplanen (2D-lista 3x3)

    Returnerar:
        En lista med tupler (rad, kol) för varje tom ruta.
        Om planen är full returneras en tom lista [].
    """
    tomma = []  # Börja med en tom lista

    # Loopa igenom varje rad (0, 1, 2)
    for rad in range(BOARD_SIZE):
        # Loopa igenom varje kolumn (0, 1, 2)
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:  # 0 = tom cell
                tomma.append((rad, kol))  # Lägg till positionen som ett par

    return tomma  # Returnera listan med alla tomma positioner


# ----------------------------------------------------------
# TEST – prova funktionen på en halvfylld spelplan
# ----------------------------------------------------------
test_brade = [
    [1, 0, 2],   # Rad 0: X är på (0,0), tomt på (0,1), O på (0,2)
    [0, 1, 0],   # Rad 1: tomt (1,0), X på (1,1), tomt (1,2)
    [2, 0, 0]    # Rad 2: O på (2,0), tomt (2,1), tomt (2,2)
]

print("🗺️ Spelplanen:")
visa_braede(test_brade)

tomma = find_empty_cells(test_brade)
print(f"🔍 Tomma celler ({len(tomma)} st): {tomma}")
print()
print("Förklaring:")
for pos in tomma:
    rad, kol = pos
    print(f"  - Position {pos}: rad {rad}, kolumn {kol}")


---
## 🎲 Del 3 – Den slumpmässiga agenten: `random_agent()`

Nu har vi `find_empty_cells()`. Nu kan vi bygga själva agenten!

### Plan:
1. Anropa `find_empty_cells()` för att få alla lediga rutor
2. Om det finns lediga rutor → välj en **på slump** med `random.choice()`
3. Returnera det valda draget (som en tuple `(rad, kol)`)

### Vad är `random.choice()`?
`random.choice(lista)` väljer ett **slumpmässigt element** från en lista.

```python
import random
frukter = ["äpple", "banan", "körsbär"]
val = random.choice(frukter)   # Kan bli vad som helst!
print(val)
```

Varje gång du kör koden kan du få ett annat svar! 🎲


In [ ]:
# ============================================================
# random_agent – väljer ett slumpmässigt drag
# ============================================================

def random_agent(board):
    """
    Väljer ett slumpmässigt drag från de lediga rutorna.

    Argument:
        board: Spelplanen (2D-lista 3x3)

    Returnerar:
        En tuple (rad, kol) med det valda draget,
        eller None om det inte finns några lediga rutor.
    """
    # Steg 1: Hitta alla tomma celler
    tomma_celler = find_empty_cells(board)

    # Steg 2: Kolla om det finns något att välja
    if tomma_celler:
        # Steg 3: Välj en slumpmässigt och returnera
        return random.choice(tomma_celler)

    # Om listan är tom (spelplanen full) returnera None
    return None  # Inga lediga rutor kvar


# ----------------------------------------------------------
# TEST – kör agenten flera gånger och se olika resultat
# ----------------------------------------------------------
test_brade2 = [
    [1, 0, 2],
    [0, 1, 0],
    [2, 0, 0]
]

print("🗺️ Spelplan:")
visa_braede(test_brade2)

print("🎲 Kör random_agent() fem gånger:")
print("  (Observera att svaren kan variera!)")
print()
for i in range(5):
    drag = random_agent(test_brade2)
    print(f"  Körning {i+1}: agenten väljer {drag}")


---
## 🧪 Experiment – Varför varierar svaret?

I cellen nedan kör vi `random_agent()` hela **20 gånger** på samma spelplan  
och räknar hur ofta varje position väljs.

**Vad tror du?** Borde varje position väljas ungefär lika ofta? 🤔  
Kör cellen och se!


In [ ]:
# ============================================================
# Experiment: räkna hur ofta varje cell väljs
# ============================================================

from collections import Counter  # Hjälper oss räkna

# En spelplan med fem tomma celler
exp_brade = [
    [1, 0, 2],
    [0, 1, 0],
    [2, 0, 0]
]

# Kör agenten 200 gånger och spara resultaten
antal_körningar = 200
resultat = []

for _ in range(antal_körningar):
    drag = random_agent(exp_brade)
    resultat.append(drag)

# Räkna hur ofta varje cell valdes
räknare = Counter(resultat)

print(f"📊 Statistik efter {antal_körningar} körningar:")
print()

# Hitta tomma celler för att visa alla möjliga val
tomma = find_empty_cells(exp_brade)
for cell in sorted(tomma):
    antal = räknare[cell]
    procent = antal / antal_körningar * 100
    stapel = "█" * (antal // 5)   # Enkel stapel
    print(f"  Cell {cell}: {antal:3d} gånger ({procent:.1f}%) {stapel}")

print()
print("💡 Slutsats: Varje cell väljs ungefär lika ofta – det är ju slumpmässigt!")


---
## 🎮 Del 4 – Spela mot den slumpmässiga AI:n

Nu sätter vi ihop allt! Vi bygger ett interaktivt spel där:
- **Du** (Spelare 1, X) klickar på knappar för att välja ruta
- **Datorn** (Spelare 2, O) använder `random_agent()` för att välja automatiskt

### Hur det fungerar:
1. Du klickar på en knapp → din symbol placeras
2. Koden kollar om du vann
3. Om inte → datorn spelar automatiskt med `random_agent()`
4. Koden kollar om datorn vann
5. Fortsätter tills någon vinner eller det är oavgjort

Kör cellen nedan och starta matchen! 🎮


In [ ]:
# ============================================================
# Interaktivt spel: Människa mot slumpmässig AI
# ============================================================

def starta_spel_mot_random():
    """Startar ett interaktivt spel mot den slumpmässiga agenten."""

    # ----------------------------------------------------------
    # Spelstatus – variabler som håller koll på spelet
    # ----------------------------------------------------------
    brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]  # Tom spelplan
    nuvarande_spelare = 1   # Spelare 1 (människa) börjar
    spelet_slut = False

    # Symboler för varje spelare
    symboler = {1: "❌", 2: "⭕"}

    # ----------------------------------------------------------
    # Widgets – knappar och informationsetiketter
    # ----------------------------------------------------------
    knappar = {}
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            btn = widgets.Button(
                description=" ",
                layout=widgets.Layout(width="80px", height="80px"),
                style={"font_size": "28px"}
            )
            knappar[(rad, kol)] = btn

    status_etikett = widgets.Label(
        value="🎮 Ditt drag! Du är ❌",
        style={"font_size": "16px"}
    )

    nytt_spel_knapp = widgets.Button(
        description="🔄 Nytt spel",
        button_style="info",
        layout=widgets.Layout(width="150px", height="40px")
    )

    # ----------------------------------------------------------
    # uppdatera_knappar – synkroniserar knapparnas utseende
    # med spelplanens data
    # ----------------------------------------------------------
    def uppdatera_knappar():
        for (r, k), btn in knappar.items():
            val = brade[r][k]
            if val == 0:
                btn.description = " "
                btn.disabled = False
                btn.button_style = ""
            elif val == 1:
                btn.description = "❌"
                btn.disabled = True
                btn.button_style = "danger"
            else:
                btn.description = "⭕"
                btn.disabled = True
                btn.button_style = "success"

    # ----------------------------------------------------------
    # dator_drag – låter AI:n göra sitt drag
    # ----------------------------------------------------------
    def dator_drag():
        nonlocal nuvarande_spelare, spelet_slut
        drag = random_agent(brade)
        if drag:
            rad, kol = drag
            make_move(brade, rad, kol, 2)   # Spelare 2 = datorn
            uppdatera_knappar()
            vinnare = check_winner(brade)
            if vinnare:
                status_etikett.value = f"🤖 Datorn vinner! {symboler[vinnare]}"
                spelet_slut = True
                for btn in knappar.values():
                    btn.disabled = True
            elif not find_empty_cells(brade):
                status_etikett.value = "🤝 Oavgjort! Ingen vann."
                spelet_slut = True
            else:
                nuvarande_spelare = 1
                status_etikett.value = "🎮 Din tur! Du är ❌"

    # ----------------------------------------------------------
    # klick – hanterar ditt klick på en ruta
    # ----------------------------------------------------------
    def klick(btn_info, rad, kol):
        nonlocal nuvarande_spelare, spelet_slut
        if spelet_slut or nuvarande_spelare != 1:
            return   # Ignorera klick om spelet är slut eller fel tur
        if make_move(brade, rad, kol, 1):   # Spelare 1 = du
            uppdatera_knappar()
            vinnare = check_winner(brade)
            if vinnare:
                status_etikett.value = f"🎉 Du vinner! {symboler[vinnare]}"
                spelet_slut = True
                for btn in knappar.values():
                    btn.disabled = True
            elif not find_empty_cells(brade):
                status_etikett.value = "🤝 Oavgjort! Ingen vann."
                spelet_slut = True
            else:
                nuvarande_spelare = 2
                status_etikett.value = "🤖 Datorn tänker..."
                dator_drag()   # Datorn spelar direkt efter dig

    # Koppla klickfunktioner till knappar
    for (r, k), btn in knappar.items():
        btn.on_click(lambda b, rad=r, kol=k: klick(b, rad, kol))

    # ----------------------------------------------------------
    # Nytt spel – återställer allt
    # ----------------------------------------------------------
    def nytt_spel(b):
        nonlocal brade, nuvarande_spelare, spelet_slut
        brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
        nuvarande_spelare = 1
        spelet_slut = False
        uppdatera_knappar()
        status_etikett.value = "🎮 Ditt drag! Du är ❌"

    nytt_spel_knapp.on_click(nytt_spel)

    # ----------------------------------------------------------
    # Bygg layouten och visa
    # ----------------------------------------------------------
    rader_layout = []
    for rad in range(BOARD_SIZE):
        rad_knappar = [knappar[(rad, kol)] for kol in range(BOARD_SIZE)]
        rader_layout.append(widgets.HBox(rad_knappar))

    spelplan_box = widgets.VBox(rader_layout)
    hela_spelet = widgets.VBox([
        widgets.Label("🎮 Tre i rad – Du (❌) mot Slumpmässig AI (⭕)"),
        spelplan_box,
        status_etikett,
        nytt_spel_knapp
    ])
    display(hela_spelet)

# Starta spelet!
starta_spel_mot_random()


---
## 📊 Del 5 – Slumpmässig vs Slumpmässig: Vem vinner?

Nu gör vi något roligt: vi låter **datorn spela mot sig självt** – två slumpmässiga agenter!

Vi kör **1 000 spel** automatiskt och räknar:
- Hur många gånger vann Spelare 1 (❌)?
- Hur många gånger vann Spelare 2 (⭕)?
- Hur många gånger blev det oavgjort?

**Fundera:** Borde det bli jämnt? Har den som börjar (Spelare 1) en fördel? 🤔


In [ ]:
# ============================================================
# Simulering: Slumpmässig agent mot slumpmässig agent
# ============================================================

def simulera_spel(antal_spel):
    """
    Simulerar ett antal spel mellan två slumpmässiga agenter.

    Argument:
        antal_spel: Hur många spel ska spelas

    Returnerar:
        En ordbok med antal vinster för varje spelare och oavgjort
    """
    resultat = {1: 0, 2: 0, "oavgjort": 0}   # Räknare

    for _ in range(antal_spel):
        # Ny tom spelplan för varje spel
        brade = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
        nuvarande = 1   # Spelare 1 börjar alltid

        # Spela tills spelet är slut
        while True:
            # Agenten väljer ett drag
            drag = random_agent(brade)

            if drag is None:
                # Inga drag möjliga – oavgjort
                resultat["oavgjort"] += 1
                break

            # Gör draget
            rad, kol = drag
            make_move(brade, rad, kol, nuvarande)

            # Kolla om vi har en vinnare
            vinnare = check_winner(brade)
            if vinnare:
                resultat[vinnare] += 1
                break

            # Kolla om spelplanen är full (oavgjort)
            if not find_empty_cells(brade):
                resultat["oavgjort"] += 1
                break

            # Byt spelare: 1→2 eller 2→1
            nuvarande = 2 if nuvarande == 1 else 1

    return resultat


# ----------------------------------------------------------
# Kör simuleringen med 1000 spel
# ----------------------------------------------------------
antal = 1000
print(f"🎮 Simulerar {antal} spel – vänta lite...")
stats = simulera_spel(antal)

print(f"\n📊 Resultat efter {antal} spel:")
print(f"  ❌ Spelare 1 vann:  {stats[1]:4d} gånger ({stats[1]/antal*100:.1f}%)")
print(f"  ⭕ Spelare 2 vann:  {stats[2]:4d} gånger ({stats[2]/antal*100:.1f}%)")
print(f"  🤝 Oavgjort:        {stats['oavgjort']:4d} gånger ({stats['oavgjort']/antal*100:.1f}%)")
print()

# Enkel visualisering med staplar
total = antal
print("📈 Visuell fördelning:")
for nyckel, etikett in [(1, "❌ Spelare 1"), (2, "⭕ Spelare 2"), ("oavgjort", "🤝 Oavgjort ")]:
    bredd = int(stats[nyckel] / total * 50)
    print(f"  {etikett}: {'█' * bredd} {stats[nyckel]}")


---
## 🧠 Del 6 – Analys: Vad är bra och dåligt med slumpmässig AI?

### ✅ Fördelar med slumpmässig agent

| Fördel | Förklaring |
|--------|-----------|
| **Enkel kod** | Bara några rader, lätt att förstå |
| **Fungerar alltid** | Den fastnar aldrig – väljer alltid ett giltigt drag |
| **Oförutsägbar** | Svår att lista ut – spelar aldrig likadant |
| **Rättvis** | Varken spelare 1 eller 2 gynnas av koden |

### ❌ Nackdelar med slumpmässig agent

| Nackdel | Förklaring |
|---------|-----------|
| **Ingen strategi** | Förstår inte spelet alls |
| **Missar vinster** | Kan ha ett vinnardrag men inte ta det |
| **Blockerar inte** | Ser inte om motspelaren är på väg att vinna |
| **Väljer inte bäst plats** | Mitten och hörnen är strategiskt bättre – men det vet inte agenten |

### 💬 Diskussionsfrågor
1. Om du spelar mot den slumpmässiga agenten, hur lätt är det att vinna?
2. Vad tror du behöver läggas till för att göra en "smart" agent?
3. Varför tror du att Spelare 1 kan ha en liten fördel i simuleringen?


---
## ✏️ Övningar

Svara på frågorna nedan genom att redigera cellerna (dubbelklicka) eller köra kodcellerna.


### 📝 Övning 1 – Spåra find_empty_cells()

Spelplanen ser ut så här:
```
2 1 2
1 0 1
0 2 0
```

**a)** Gå igenom koden i `find_empty_cells()` manuellt, rad för rad.  
Skriv ner vilka positioner `(rad, kol)` som läggs in i `tomma`-listan i ordning.

*Skriv ditt svar här (dubbelklicka för att redigera):*  
Svar: ...

**b)** Kör koden nedan för att kontrollera ditt svar.


In [ ]:
# Övning 1b – kontrollera ditt svar
brade_ex1 = [
    [2, 1, 2],
    [1, 0, 1],
    [0, 2, 0]
]
print("Spelplan:")
visa_braede(brade_ex1)
tomma_ex1 = find_empty_cells(brade_ex1)
print(f"Tomma celler: {tomma_ex1}")


### 📝 Övning 2 – Varför random.choice()?

Titta på `random_agent()`. Den använder `random.choice(tomma_celler)`.

**a)** Vad skulle hända om vi använde `tomma_celler[0]` istället?  
(dvs. alltid tar den FÖRSTA tomma cellen)

*Skriv ditt svar här:*  
Svar: ...

**b)** Vad skulle vara nackdelen med `tomma_celler[0]`?  
Tänk på: Kan motspelaren lista ut agentens drag i förväg?

*Skriv ditt svar här:*  
Svar: ...


### 📝 Övning 3 – Vad händer när spelplanen är full?

**a)** Vad returnerar `find_empty_cells()` om spelplanen är **helt full** (inga 0)?

*Skriv ditt svar här:*  
Svar: ...

**b)** Vad returnerar `random_agent()` i det fallet?

*Skriv ditt svar här:*  
Svar: ...

**c)** Kör koden nedan för att kontrollera:


In [ ]:
# Övning 3c – full spelplan
full_brade = [
    [1, 2, 1],
    [2, 1, 2],
    [2, 1, 2]
]
print("Full spelplan:")
visa_braede(full_brade)

tomma_full = find_empty_cells(full_brade)
print(f"find_empty_cells returnerar: {tomma_full}")

drag_full = random_agent(full_brade)
print(f"random_agent returnerar: {drag_full}")


### 📝 Övning 4 – Modifiera agenten: föredra mitten

Den nuvarande `random_agent()` väljer helt på slump.  
Men mitten (rad 1, kolumn 1) är strategiskt bäst i Tre i rad!

**Uppgift:** Modifiera `random_agent()` så att den:
1. Kollar om mitten (1,1) är tom
2. Om den är tom → välj mitten
3. Annars → välj slumpmässigt som vanligt

Fyll i koden nedan:


In [ ]:
# Övning 4 – Modifiera random_agent att föredra mitten

def random_agent_mitten(board):
    """
    Förbättrad agent som föredrar mitten.
    Fyll i den saknade koden!
    """
    # Kolla om mitten är ledig
    if board[1][1] == 0:
        # TODO: Returnera mitten-positionen
        pass   # ← Byt ut pass mot rätt kod

    # Annars: välj slumpmässigt som vanligt
    tomma_celler = find_empty_cells(board)
    if tomma_celler:
        return random.choice(tomma_celler)
    return None


# Test
test_brade3 = [
    [0, 0, 0],
    [0, 0, 0],   # Mitten (1,1) är tom
    [0, 0, 0]
]
print("Test 1 – Mitten ledig:")
drag = random_agent_mitten(test_brade3)
print(f"  Agent väljer: {drag}  ← Ska vara (1, 1)")

test_brade4 = [
    [0, 0, 0],
    [0, 1, 0],   # Mitten (1,1) är tagen
    [0, 0, 0]
]
print("\nTest 2 – Mitten tagen:")
drag2 = random_agent_mitten(test_brade4)
print(f"  Agent väljer: {drag2}  ← Ska vara något annat än (1,1)")


### 📝 Övning 5 – Räkna tomma celler utan `find_empty_cells()`

**Utmaning:** Skriv en **separat funktion** som räknar hur många tomma celler det finns,  
**utan** att använda `find_empty_cells()`.

Tips: Använd en räknare-variabel och en dubbel for-loop.


In [ ]:
# Övning 5 – Räkna tomma celler utan hjälpfunktionen

def räkna_tomma(board):
    """
    Räknar antalet tomma celler (värde 0).
    Skriv din lösning nedan!
    """
    räknare = 0
    # TODO: Lägg till din dubbla for-loop här

    return räknare


# Test
test_ex5 = [
    [1, 0, 2],
    [0, 1, 0],
    [2, 0, 0]
]
antal_tomma = räkna_tomma(test_ex5)
print(f"Antal tomma celler: {antal_tomma}")
print(f"Förväntat svar: 5")
print(f"✅ Rätt!" if antal_tomma == 5 else "❌ Försök igen!")


### 📝 Övning 6 – Analysera simuleringen

Kör simuleringen med 5 000 spel (i stället för 1 000) och svara:

**a)** Vilket resultat ser du? Är det ungefär jämnt?

*Skriv ditt svar här:*  
Svar: ...

**b)** Har Spelare 1 (börjar) en statistisk fördel? Varför eller varför inte?

*Skriv ditt svar här:*  
Svar: ...

**c)** Kör koden nedan och jämför:


In [ ]:
# Övning 6c – simulera 5000 spel
stats5000 = simulera_spel(5000)
print(f"📊 Resultat efter 5000 spel:")
print(f"  ❌ Spelare 1: {stats5000[1]} ({stats5000[1]/5000*100:.1f}%)")
print(f"  ⭕ Spelare 2: {stats5000[2]} ({stats5000[2]/5000*100:.1f}%)")
print(f"  🤝 Oavgjort:  {stats5000['oavgjort']} ({stats5000['oavgjort']/5000*100:.1f}%)")


---
## 🧪 Quiz – Testa dina kunskaper!

Kör varje cell för att svara och få feedback.


In [ ]:
# Quiz 1: Vad returnerar find_empty_cells() på en tom spelplan?
svar_1 = None  # Ändra None till ditt svar!

# Alternativ:
# A = "En tom lista []"
# B = "En lista med 9 tupler"
# C = "Talet 9"
# D = "None"

svar_1 = "B"   # ← Ändra här!

if svar_1 == "B":
    print("✅ Rätt! En tom spelplan har 9 tomma celler, så listan innehåller 9 tupler (rad,kol).")
else:
    print("❌ Försök igen! Tänk på: varje tom cell lägger till en tuple i listan.")


In [ ]:
# Quiz 2: Vad gör random.choice(lista)?
svar_2 = None

# A = "Väljer det första elementet"
# B = "Sorterar listan"
# C = "Väljer ett slumpmässigt element"
# D = "Räknar listans längd"

svar_2 = "C"   # ← Ändra här!

if svar_2 == "C":
    print("✅ Rätt! random.choice() väljer ett slumpmässigt element från listan.")
else:
    print("❌ Försök igen! Ledtråd: tänk på vad 'random' (slumpmässig) betyder.")


In [ ]:
# Quiz 3: Vad returnerar random_agent() om spelplanen är full?
svar_3 = None

# A = "En slumpmässig cell utanför planen"
# B = "0"
# C = "None"
# D = "Den kastar ett fel (crash)"

svar_3 = "C"   # ← Ändra här!

if svar_3 == "C":
    print("✅ Rätt! Om find_empty_cells() returnerar [], är if-satsen False och None returneras.")
else:
    print("❌ Fel! Titta på sista raden i random_agent(): vad returneras om listan är tom?")


In [ ]:
# Quiz 4: Varför är en slumpmässig agent ett bra startpunkt?
svar_4 = None

# A = "Den spelar alltid perfekt"
# B = "Den är enkel att förstå och garanterar alltid ett giltigt drag"
# C = "Den lär sig med tiden"
# D = "Den är omöjlig att slå"

svar_4 = "B"   # ← Ändra här!

if svar_4 == "B":
    print("✅ Rätt! Slumpmässig agent är enkel, fungerar alltid och är en bra grund att bygga vidare på.")
else:
    print("❌ Fel! Tänk på fördelarna vi gick igenom i Del 6.")


In [ ]:
# Quiz 5: Vilket påstående OM den slumpmässiga agenten är SANT?
svar_5 = None

# A = "Den blockerar alltid motspelaren"
# B = "Den väljer alltid mitten"
# C = "Den missar ibland ett vinnardrag"
# D = "Den spelar optimalt"

svar_5 = "C"   # ← Ändra här!

if svar_5 == "C":
    print("✅ Rätt! Eftersom agenten väljer slumpmässigt kan den missa ett vinnardrag.")
else:
    print("❌ Fel! Kom ihåg nackdelarna med slumpmässig agent från Del 6.")


---
## 🎓 Sammanfattning – Lektion 3

### Vad lärde vi oss?

✅ **En agent** är ett program som observerar sin omgivning och tar beslut  
✅ **find_empty_cells()** loopar igenom spelplanen och returnerar en lista med tomma positioner  
✅ **random_agent()** väljer ett slumpmässigt drag från de tomma cellerna med `random.choice()`  
✅ **Slumpmässig vs slumpmässig** – ungefär jämt, med en liten fördel för den som börjar  

### Nya funktioner du kan:
| Funktion | Vad den gör |
|----------|------------|
| `find_empty_cells(board)` | Returnerar lista med tomma `(rad,kol)` |
| `random_agent(board)` | Väljer ett slumpmässigt drag |
| `simulera_spel(n)` | Kör n spel automatiskt och räknar statistik |

### 🚀 Nästa lektion
I Lektion 4 bygger vi en **smart agent** med regler:
- Regel 1: Vinn om möjligt 🏆
- Regel 2: Blockera motspelaren 🛡️
- Regel 3: Ta mitten ⭐

Vi jämför sedan den smarta agenten mot den slumpmässiga – vem vinner? 🤖⚔️🎲
